## Import Datas

In [ ]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/ICATH 2026/code/data/preprocessed options datas/"

Mounted at /content/drive


In [2]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
import numpy as np
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
TechnologyUnderlying = [ "AAPL", "AMZN", "GOOGL", "META", "MSFT" ]
AutomobileUnderlying = [ "F", "GM", "LCID", "RIVN", "TSLA" ]

listTickers = AutomobileUnderlying + TechnologyUnderlying

In [ ]:
dataset = {
    "AAPL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "AMZN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GOOGL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "META" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MSFT" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "F" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "LCID" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "RIVN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "TSLA" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
}

In [5]:
for ticker in dataset.keys():
    train_path = os.path.join(folder, f"{ticker}_train.csv")
    test_path = os.path.join(folder, f"{ticker}_test.csv")

    if not os.path.exists(train_path):
        print(f"Missing: {train_path}")
        continue

    train_dataset = pd.read_csv(train_path)
    test_dataset = pd.read_csv(test_path)

    train_dataset = train_dataset.fillna(train_dataset.mean(numeric_only=True))
    test_dataset = test_dataset.fillna(test_dataset.mean(numeric_only=True))

    dataset[ticker]["train"] = train_dataset
    dataset[ticker]["test"] = test_dataset

## Define functions

In [6]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.9 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import xgboost as xgb
from xgboost import XGBRegressor
import optuna

In [ ]:
test_size=0.2
random_state=42
target = 'lastPrice'
list_histos_datas_inputs = ["strike", "impliedVolatility", "timeToMaturity",
                            "riskFreeRate", "underlyingLastPrice"]

feature_combinations = [
    [],

    ["GS_" + ticker], ["SC_" + ticker], ["VIX"], ["PCR"],

    ["GS_" + ticker , "SC_" + ticker], ["GS_" + ticker , "VIX"], ["GS_" + ticker , "PCR"], ["SC_" + ticker , "VIX"], ["SC_" + ticker , "PCR"], ["VIX" , "PCR"],

    ["GS_" + ticker , "SC_" + ticker , "VIX"], ["GS_" + ticker , "SC_" + ticker , "PCR"], ["GS_" + ticker , "VIX" , "PCR"], ["SC_" + ticker , "VIX" , "PCR"],

	["GS_" + ticker , "SC_" + ticker , "VIX" , "PCR"]
]

In [ ]:
def train_xgboost_with_optuna(X_train, y_train, X_test, y_test, best_params):

    params = best_params.copy()
    params["objective"] = "reg:squarederror"
    params["eval_metric"] = "rmse"
    params["tree_method"] = "hist"

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=500,
        evals=[(dtrain, "train")],
        early_stopping_rounds=50,
        verbose_eval=False
    )

    preds = model.predict(dtest)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    nrmse = rmse / np.mean(y_test)

    return mae, rmse, nrmse


def objective_for_xgboost(trial, X_train, y_train):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 0, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5)
    }

    model = XGBRegressor(**params)
    score = cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_squared_error")

    return -score.mean()

def predict_current_price_using_xgboost(option_type, ticker, list_histos_datas_inputs=list_histos_datas_inputs, feature_combinations=feature_combinations):
  for proxy in feature_combinations:
    train_dataset = dataset[ticker]["train"]
    train_dataset = train_dataset[train_dataset["optionType"] == option_type]
    X_train = train_dataset[list_histos_datas_inputs + proxy].values
    y_train = train_dataset[["lastPrice"]].values

    test_dataset = dataset[ticker]["test"]
    test_dataset = test_dataset[test_dataset["optionType"] == option_type]
    X_test = test_dataset[list_histos_datas_inputs+ proxy].values
    y_test = test_dataset[["lastPrice"]].values

    #Normaliser les dataset
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.fit_transform(X_test)

    study = study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: objective_for_xgboost(trial, X_train, y_train), n_trials=10)
    best_params = study.best_params

    mae, rmse, nrmse = train_xgboost_with_optuna(
        X_train, y_train,
        X_test, y_test,
        best_params
    )
    print(f"{proxy} for {ticker} => (MAE={mae:.3f}; RMSE={rmse:.3f}; ; NRMSE={nrmse:.3f})")


## Price options using extreme gradient boosting machine

In [19]:
ticker = "RIVN"

In [21]:
predict_current_price_using_xgboost("put", ticker)

[] for RIVN => (MAE=0.317; RMSE=0.497; ; NRMSE=0.266)
['GS_RIVN'] for RIVN => (MAE=0.318; RMSE=0.503; ; NRMSE=0.268)
['SC_RIVN'] for RIVN => (MAE=0.316; RMSE=0.508; ; NRMSE=0.271)
['VIX'] for RIVN => (MAE=0.316; RMSE=0.504; ; NRMSE=0.269)
['PCR'] for RIVN => (MAE=0.317; RMSE=0.497; ; NRMSE=0.266)
['GS_RIVN', 'SC_RIVN'] for RIVN => (MAE=0.294; RMSE=0.432; ; NRMSE=0.230)
['GS_RIVN', 'VIX'] for RIVN => (MAE=0.291; RMSE=0.423; ; NRMSE=0.226)
['GS_RIVN', 'PCR'] for RIVN => (MAE=0.286; RMSE=0.410; ; NRMSE=0.219)
['SC_RIVN', 'VIX'] for RIVN => (MAE=0.299; RMSE=0.449; ; NRMSE=0.240)
['SC_RIVN', 'PCR'] for RIVN => (MAE=0.299; RMSE=0.447; ; NRMSE=0.239)
['VIX', 'PCR'] for RIVN => (MAE=0.290; RMSE=0.420; ; NRMSE=0.224)
['GS_RIVN', 'SC_RIVN', 'VIX'] for RIVN => (MAE=0.307; RMSE=0.496; ; NRMSE=0.265)
['GS_RIVN', 'SC_RIVN', 'PCR'] for RIVN => (MAE=0.305; RMSE=0.496; ; NRMSE=0.265)
['GS_RIVN', 'VIX', 'PCR'] for RIVN => (MAE=0.304; RMSE=0.482; ; NRMSE=0.257)
['SC_RIVN', 'VIX', 'PCR'] for RIVN => (MAE=

In [23]:
predict_current_price_using_xgboost("call", ticker)

[] for RIVN => (MAE=0.269; RMSE=0.417; ; NRMSE=0.166)
['GS_RIVN'] for RIVN => (MAE=0.218; RMSE=0.320; ; NRMSE=0.127)
['SC_RIVN'] for RIVN => (MAE=0.219; RMSE=0.350; ; NRMSE=0.139)
['VIX'] for RIVN => (MAE=0.217; RMSE=0.327; ; NRMSE=0.130)
['PCR'] for RIVN => (MAE=0.208; RMSE=0.295; ; NRMSE=0.117)
['GS_RIVN', 'SC_RIVN'] for RIVN => (MAE=0.223; RMSE=0.366; ; NRMSE=0.145)
['GS_RIVN', 'VIX'] for RIVN => (MAE=0.254; RMSE=0.423; ; NRMSE=0.168)
['GS_RIVN', 'PCR'] for RIVN => (MAE=0.228; RMSE=0.345; ; NRMSE=0.137)
['SC_RIVN', 'VIX'] for RIVN => (MAE=0.220; RMSE=0.352; ; NRMSE=0.140)
['SC_RIVN', 'PCR'] for RIVN => (MAE=0.219; RMSE=0.355; ; NRMSE=0.141)
['VIX', 'PCR'] for RIVN => (MAE=0.211; RMSE=0.316; ; NRMSE=0.126)
['GS_RIVN', 'SC_RIVN', 'VIX'] for RIVN => (MAE=0.189; RMSE=0.339; ; NRMSE=0.135)
['GS_RIVN', 'SC_RIVN', 'PCR'] for RIVN => (MAE=0.188; RMSE=0.336; ; NRMSE=0.134)
['GS_RIVN', 'VIX', 'PCR'] for RIVN => (MAE=0.186; RMSE=0.294; ; NRMSE=0.117)
['SC_RIVN', 'VIX', 'PCR'] for RIVN => (MAE=